In [ ]:
"""

This is Naive RAG (Basic RAG). 
You load a document, split it into chunks, generate embeddings, 
store them in FAISS, & retrieve the most relevant chunk using similarity search 
when a user asks a question. 

The retrieved chunk is passed as context alongside 
the original question to your LLM, which generates an answer based on that context. There is no advanced query transformation, reranking, multi-step retrieval or agentic decision-making involved.
"""

In [8]:
from langchain_community.document_loaders import TextLoader
# LOAD   LOAD   LOAD   LOAD
loader = TextLoader("./notes/fruits_and_vegetables.txt")

documents = loader.load()

In [9]:
# print(documents[0].page_content)
print(documents[0].metadata)

{'source': './notes/fruits_and_vegetables.txt'}


In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
#   SPLIT   SPLIT   SPLIT   SPLIT
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=1
)

chunks = text_splitter.split_documents(documents)
# print(chunks[0].page_content)
# print(len(chunks))
# print(chunks[0].page_content)
# print("------------")
# print(chunks[1].page_content)

In [11]:
"""
# EMBEDDING into Vector   EMBEDDING into Vector  EMBEDDING into Vector
    
    THIS snippet here is only for learning and debugging purpose. 
    It is not necessary to run this one to create a RAG pipeline
"""
from langchain_ollama import OllamaEmbeddings
embedding_model = OllamaEmbeddings(
    model="nomic-embed-text"
)

vectors = embedding_model.embed_documents(
    [doc.page_content for doc in chunks]
)

print(vectors[0])
print(len(vectors))

[0.032752205, 0.068445906, -0.15539417, -0.032033183, 0.025506075, 0.0015184266, -0.033028662, -0.063875824, 0.009901285, -0.012641162, 0.04692346, 0.01846017, 0.07768375, 0.051293626, -0.01370135, 0.00948923, -0.045861967, -0.05159239, 0.04598076, 0.06692924, -0.037889175, 0.044615496, -0.025733856, 0.0169641, 0.00603015, 0.03920874, -0.0811716, 0.013404017, -0.0061354316, -0.008897775, 0.011733257, -0.038644984, 0.022851216, 0.01936049, 0.03020469, -0.03504562, 0.051688034, 0.04396695, 0.038159426, 0.063732795, 0.044923596, 0.034283753, 0.027648734, -0.05251945, 0.02649773, -0.0075678644, 0.04592037, 0.007188063, 0.017645858, -0.036005884, 0.0010852638, 0.03883107, -0.052943073, -0.06460175, 0.058368832, -0.0038335235, 0.05585124, 0.037958402, 0.045722056, 0.009885601, 0.03881044, 0.092898145, -0.05596046, 0.11605663, 0.06444892, -0.038213383, -0.031505745, 0.09182425, -0.040825263, 0.018039657, 0.018816812, -0.06908339, 0.040518478, 0.008646403, 0.013775615, -0.01952255, 0.000499688

In [12]:
from langchain_community.vectorstores import FAISS

# CREATE-VECTOR-STORE CREATE-VECTOR-STORE  CREATE-VECTOR-STORE

# Store — create and persist
# all chunks from the doc are converted into vector using the
# embedding_model we passed & saved  
# then a mapping is maintained between the vector and the original chunk by wrapper
# mapping is stored in index.pkl
vectorstore = FAISS.from_documents(chunks, embedding_model)
vectorstore.save_local("./faiss_index")


"""
    we are trying to load vectorstore. for learning
    it is done immedately after creating the vectorstore. but in real world
    scenario, it will be done in a different script, and condition based
    too 
    
    never pass
    allow_dangerous_deserialization = true
    because it might be that this pkl file contains malicious code.
    and it might do impose security risk
    
    never pass allow_dangerous_deserialization=True
    when loading a .pkl from an untrusted or external source
    because it might contain malicious code and pose a security risk.
    when you created the index yourself, it is safe to pass it.

    here for learning purpose, we again get it into same vectorstore obj
    remember to pass same embedding_model as we used to create the vectorstore
"""
# Retrieve — load back and search
vectorstore = FAISS.load_local(
    "./faiss_index",
    embedding_model,
    allow_dangerous_deserialization=True
)


"""
    This is pure math — 
    your query string gets embedded into a vector using nomic-embed-text, 
    then FAISS computes cosine similarity between that query vector 
    and all stored vectors, and returns the top k=1 closest chunk. 


    No LLM, no AI reasoning, just geometric distance between float arrays. 

    But note that The “intelligence” here is entirely 
    from the embedding model that was trained to place semantically 
    similar text close together in vector space.

"""
results = vectorstore.similarity_search("Tell us about carrots", k=1)

for doc in results:
    print(doc.page_content)
    print("---")

In botany, a fruit develops from the flower of a plant and usually contains seeds. Apples, oranges, grapes, tomatoes, cucumbers and pumpkins are botanical fruits. A vegetable is a culinary term for other edible plant parts, such as roots, stems, leaves, flowers and bulbs.

Carrots are roots. Celery is mainly a stem. Spinach and lettuce are leaves. Broccoli is an immature flower structure. Onions are bulbs. These foods are commonly called vegetables because of how they are used in cooking.
---


In [15]:
from langchain_ollama import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

"""
    our qwen model and temperature is set to 0.2, 
    which means it will be more deterministic 
    and less random in its responses.
"""

llm = OllamaLLM(model="qwen2.5:1.5b", temperature=0.2)


prompt = ChatPromptTemplate.from_template(
    """
Answer the question based only on the context below.

Context: {context}

Question: {question}
"""
)

"""
vectorstore.as_retriever(search_kwargs={"k": 1}) converts your vector store (FAISS, for example) into a Retriever interface that LangChain can use to fetch relevant Documents based on a user's query. Here, k=1 instructs the retriever to return the single most relevant chunk for each search query. When a user asks a question, the retriever converts that question into an embedding, compares it against the stored vectors using the vector store's configured similarity search, and returns the matching Document. Think of it as a Java service wrapping your database search logic and exposing a standard retrieval method. Importantly, as_retriever() configures the retrieval mechanism; it does not itself retrieve anything until you invoke it with a query.
"""
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

"""
Without RunnablePassthrough() there would be no way to pass 
the raw question string into the prompt — 
everything in LCEL must be a Runnable object, 
a plain string isnt. '


RunnablePassthrough() is just a wrapper that 

makes your string play nicely in the pipeline.
"""

"""
This is a LangChain Expression Language (LCEL) pipeline that connects retrieval, prompt construction, LLM execution and output parsing into one chain. The first part, {"context": retriever, "question": RunnablePassthrough()}, creates a dictionary with two inputs: context invokes the retriever using the incoming question and fetches relevant document chunks, while question passes the original question unchanged. The | prompt feeds this dictionary into your prompt template, which combines the retrieved context and user's question into a prompt. | llm sends that prompt to your Ollama model for generation, and | StrOutputParser() extracts the response as a plain Python string rather than leaving it as an AI message object. Essentially, this is your RAG execution pipeline: Retrieve → Construct Prompt → Generate Answer → Parse Output.
"""

"""

RunnablePassthrough() receives the original input "Tell us about plant?" and passes it unchanged into the "question" field.

At the same time, the retriever receives that same input and searches your FAISS vector store for relevant chunks, which become the "context" value.

So both receive the same original question, but perform different responsibilities:
- retriever → Finds relevant document chunks.
- RunnablePassthrough() → Preserves the original question.
Both values are then passed to prompt.


The name means it passes the input through unchanged.

One correction: it doesn't directly pass the question to both the LLM and vector store.
- Retriever (Vector Store): Uses the question to retrieve relevant chunks.
- RunnablePassthrough: Preserves the original question for the prompt, which eventually goes to the LLM.
So the same input takes two paths, and both are brought together in the prompt.


"""
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

response = chain.invoke("Tell us about carrot")
print(response)



A carrot is a root vegetable that is botanically classified as a fruit. It develops from a flower and contains seeds. While it can be eaten raw, it is typically used in salads and is often treated as a vegetable in everyday cooking.


In [16]:
# import pickle

# with open("./faiss_index/index.pkl", "rb") as f:
#     data = pickle.load(f)

# print(data)